# 문항 1 네이버 연관검색어 수집 함수 만들기

In [67]:
import requests
import json

def get_related_keywords(keyword) -> list[str]:
    URL = "https://ac.search.naver.com/nx/ac"
    PARAMS = {
        "q" : keyword,
        "_callback" : "_jsonp_8",
        "r_enc" : "UTF-8",
        "st" : "100"
    }
    HEADERS = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    }
    try:
        response = requests.get(URL, params=PARAMS, headers=HEADERS, timeout=10)
        response.raise_for_status()
        txt = response.text
        txt_dict = txt.split('(')[1].split(')')[0]
        txt_json = json.loads(txt_dict)
        searches = txt_json['items'][0]
        result = [search[0] for search in searches]
        return result
    except:
        return []

get_related_keywords("부트캠프")

['부트캠프',
 '부트캠프 뜻',
 '부트캠프 취업',
 'ai 부트캠프',
 '직무부트캠프',
 '코햄 부트캠프',
 '마케팅 부트캠프',
 '맥북 부트캠프',
 '코멘토 부트캠프',
 '넷플릭스 부트캠프']

# 문항 2 네이버 웹툰 전체 목록 수집

In [ ]:
import pandas as pd
import requests

def get_webtoon_list() -> None:
    URL = "https://comic.naver.com/api/webtoon/titlelist/weekday"
    PARAMS = {
        "order" : "user",
    }
    HEADERS = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    }

    days = {
        "MONDAY" : "월요일",
        "TUESDAY" : "화요일",
        "WEDNESDAY" : "수요일",
        "THURSDAY" : "목요일",
        "FRIDAY" : "금요일",
        "SATURDAY" : "토요일",
        "SUNDAY" : "일요일",
    }
    result = {
        "제목" : [],
        "링크" : [],
        "요일" : [],
    }

    response = requests.get(URL, params=PARAMS, headers=HEADERS, timeout=10)
    data = response.json()['titleListMap']
    
    for day in data:
        for webtoon in data[day]:
            result["제목"].append(webtoon['titleName'])
            result["링크"].append("https://comic.naver.com/webtoon/list?titleId=" + str(webtoon['titleId']))
            result["요일"].append(days[day])

    df = pd.DataFrame(result)
    df.to_csv("naver_webtoon.csv", index=False, encoding="utf-8-sig")

    return None

get_webtoon_list()

# 문항 3 사람인 채용공고 10페이지 수집

In [39]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup

def null_searcher(node, selector):
    try:
        return node.select_one(selector).text.strip()
    except:
        return ""

def job_post_search(i) -> None:
    delay = 0.5
    URL = "https://www.saramin.co.kr/zf_user/jobs/public/list"
    HEADERS = {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
            }

    result = {
        "기업명" : [],
        "그룹사" : [],
        "기업종류" : [],
        "공고명" : [],
        "직무키워드" : [],
        "학력" : [],
        "경력구분" : [],
        "근무지" : [],
    }
    
    for page in range(1, i+1):
        PARAMS = {
            "page" : page,
            "isAjaxRequest" : "y",
        }

        soup = BeautifulSoup(requests.get(URL, params=PARAMS, headers=HEADERS, timeout=10).text, "html.parser")
        jobs = soup.find_all("div", class_="list_item")

        for job in jobs:
            keywords = [keyword.text.strip() for keyword in job.select("span.job_sector span")]
            
            result["기업명"].append(null_searcher(job, "a.str_tit"))
            result["그룹사"].append(null_searcher(job, "span.main_corp"))
            result["기업종류"].append(null_searcher(job, "span.info_stock"))
            result["공고명"].append(null_searcher(job, "div.job_tit > span"))
            result["직무키워드"].append(keywords)
            result["학력"].append(null_searcher(job, "p.education"))
            result["경력구분"].append(null_searcher(job, "p.career"))
            result["근무지"].append(null_searcher(job, "p.work_place"))

        time.sleep(delay)

    df = pd.DataFrame(result)
    df.to_csv("saramin.csv", index=False, encoding="utf-8")
    return None

job_post_search(10)

기업명이 a tag에 들어가있는 것이 있고 span tag에 들어가 있는것이 있어 완벽하게 추출하는게 불가능해 보임.